# 1.0 Importação das Bibliotecas Usadas e do Dataset

In [9]:
import pandas as pd
import numpy as np
import time
from sklearn import model_selection
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report
from sklearn.model_selection import GridSearchCV

In [2]:
data = pd.read_csv("dataset_tratado.csv")
data

,Unnamed: 0,Age,CDR,SES,nWBV,Visit,MMSE,ASF,eTIV,Group,MR Delay
0,0,87.0,0.0,2.0,0.696106,1.0,27.0,0.883440,1986.550000,0.0,0.0
1,1,88.0,0.0,2.0,0.681062,2.0,30.0,0.875539,2004.479526,0.0,457.0
2,2,75.0,0.5,1.8,0.736336,1.0,23.0,1.045710,1678.290000,1.0,0.0
3,3,76.0,0.5,1.6,0.713402,2.0,28.0,1.010000,1737.620000,1.0,560.0
4,4,80.0,0.5,2.6,0.701236,3.0,22.0,1.033623,1697.911134,1.0,1895.0
...,...,...,...,...,...,...,...,...,...,...,...
368,368,82.0,0.5,1.0,0.693926,2.0,28.0,1.036690,1692.880000,1.0,842.0
369,369,86.0,0.5,1.0,0.675457,3.0,26.0,1.039686,1688.009649,1.0,2297.0
370,370,61.0,0.0,2.0,0.801006,1.0,30.0,1.330540,1319.020000,0.0,0.0
371,371,63.0,0.0,2.0,0.795981,2.0,30.0,1.322890,1326.650000,0.0,763.0


# 2.0 Pre-processamento dos Dados

## 2.1 Criando X e y

In [3]:
X = np.array(data.drop(['Group', 'Unnamed: 0'], axis=1))
y = np.array(data['Group'])

## 2.2 Separando em Treino e Teste

In [4]:
RANDOM_STATE = 42

X_train, X_test, y_train, y_test = model_selection.train_test_split(X, y, random_state=RANDOM_STATE, test_size=0.2)

# 3.0 Modelagem dos Dados

In [5]:
def Modelagem(X_train, y_train, X_test, Modelo):
    
    pca_pipeline = Pipeline([
        ('scaler', StandardScaler()),        
        ('pca', PCA(n_components=0.95)),      
        ('modelo', Modelo)   
    ])
    
    pca_pipeline.fit(X_train, y_train)
    
    previsoes = pca_pipeline.predict(X_test)

    return previsoes

In [6]:
def ModelagemRC(X_train, y_train, X_test, Modelo):
    
    Modelo.fit(X_train, y_train)
    
    return Modelo.predict(X_test)

In [7]:
def ModelagemRL(X_train, y_train, X_test, Solver):
    
    pipeline = Pipeline([
        ('scaler', StandardScaler()),        # Normalização essencial para solvers de gradiente 
        ('pca', PCA(n_components=0.95)),     # Redução de dimensionalidade conforme o artigo 
        ('modelo', LogisticRegression(solver=Solver, max_iter=1000, tol=1e-5))   
    ])
    
    start_time = time.perf_counter()
    pipeline.fit(X_train, y_train)
    end_time = time.perf_counter()
    
    previsoes = pipeline.predict(X_test)
    tempo_execucao = end_time - start_time
    
    return previsoes, tempo_execucao

In [10]:
previsoes_rl, tempo = ModelagemRL(X_train, y_train, X_test, 'lbfgs')

print(f"Tempo de execução com solver L-BFGS: {tempo:.4f} segundos\n")
print("=== Relatório de Métricas (Regressão Logística) ===")
print(classification_report(y_test, previsoes_rl))

Tempo de execução com solver L-BFGS: 0.1400 segundos

=== Relatório de Métricas (Regressão Logística) ===
              precision    recall  f1-score   support

         0.0       1.00      0.95      0.98        43
         1.0       0.94      1.00      0.97        32

    accuracy                           0.97        75
   macro avg       0.97      0.98      0.97        75
weighted avg       0.97      0.97      0.97        75



# 4.0 Modelos de Aprendizagem

## 4.1 Random Forest

Treinando o modelo Random Forest utilizando os hiperparâmetros default

In [7]:
modelo = RandomForestClassifier(random_state=42) # inicializando o modelo com random_state = 42

random_forest_predict = ModelagemRC(X_train, y_train, X_test, modelo) # treinando o modelo e realizando o predict

print(classification_report(y_test, random_forest_predict)) # exibindo os resultados

              precision    recall  f1-score   support

         0.0       0.98      0.95      0.96        43
         1.0       0.94      0.97      0.95        32

    accuracy                           0.96        75
   macro avg       0.96      0.96      0.96        75
weighted avg       0.96      0.96      0.96        75



Apesar da acurácia obtida ser semelhante à acurácia exibida no artigo, utilizamos o GridSearchCV da biblioteca scikit-learn para realizar a busca dos melhores hiperparâmetros por meio de cross validation.

In [8]:
# gera valores igualmente espaçados para testar diferentes quantidades de árvores na floresta
n_estimators = [int(x) for x in np.linspace(start=100, stop = 200, num=10)]
# define as estratégias de seleção de atributos em cada divisão da árvore
max_features = ['sqrt', 'log2']
# gera valores igualmente espaçados para controlar a profundidade máxima das árvores
max_depth = [int(x) for x in np.linspace(start=10, stop = 40, num=4)]
max_depth.append(None)
# gera valores para o número mínimo de amostras em cada folha
min_samples_leaf = [int(x) for x in np.linspace(start=1, stop = 3, num=3)]

param_grid = {
    'n_estimators' : n_estimators,
    'max_features' : max_features,
    'min_samples_leaf' : min_samples_leaf,
    'max_depth' : max_depth,
}

In [ ]:
random_forest = RandomForestClassifier(random_state=42) # inicializando o modelo com random_state = 42

cv_random_forest = GridSearchCV(estimator=random_forest, param_grid=param_grid, cv = 5, verbose=2, n_jobs=-1)

cv_random_forest.fit(X_train, y_train) # realizando o gridseachcv

best_model = cv_random_forest.best_estimator_ # armazena o melhor modelo encontrado

# exibindo os resultados associado ao melhor modelo encontrado

print('Ein: %0.4f' % (1 - accuracy_score(y_train, best_model.predict(X_train))))
print('Eout: %0.4f' % (1 - accuracy_score(y_test, best_model.predict(X_test))))
print(classification_report(y_test, best_model.predict(X_test)))

print("Melhor Modelo:", best_model)

Fitting 5 folds for each of 300 candidates, totalling 1500 fits


Após a execução do GridSearchCV, observamos que melhores hiperparâmetros para o RandomForestClassifier são max_depth (profundidade máxima) = 10 e min_samples_leaf (mínimo de amostras por folha) = 2.

In [ ]:
modelo = RandomForestClassifier(max_depth=10, min_samples_leaf=2, random_state=42) # inicializando o modelo com os hiperparametros encontrados 

random_forest_predict = ModelagemRC(X_train, y_train, X_test, modelo) # treinando o modelo e realizando o predict

print(classification_report(y_test, random_forest_predict)) # exibindo os resultados

              precision    recall  f1-score   support

         0.0       0.98      0.95      0.96        43
         1.0       0.94      0.97      0.95        32

    accuracy                           0.96        75
   macro avg       0.96      0.96      0.96        75
weighted avg       0.96      0.96      0.96        75

